In [1]:
# imports

import os
from dotenv import load_dotenv
from huggingface_hub import login
from pricer.evaluator import evaluate
from litellm import completion
from pricer.items import Item
import numpy as np
from tqdm.notebook import tqdm
import csv
from sklearn.feature_extraction.text import HashingVectorizer
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from torch.optim.lr_scheduler import CosineAnnealingLR


In [29]:
LITE_MODE = False

load_dotenv(override=True)
hf_token = os.environ['HUGGING_FACE_KEY']
login(hf_token, add_to_git_credential=True)

In [30]:
username = "ed-donner"
dataset = f"{username}/items_lite" if LITE_MODE else f"{username}/items_full"

train, val, test = Item.from_hub(dataset)

print(f"Loaded {len(train):,} training items, {len(val):,} validation items, {len(test):,} test items")

Loaded 800,000 training items, 10,000 validation items, 10,000 test items


## HUMAN PREDICTIOn

In [6]:
# Write the test set to a CSV

with open('human_in.csv', 'w', encoding="utf-8") as csvfile:
    writer = csv.writer(csvfile)
    for t in test[:100]:
        writer.writerow([t.summary, 0])

In [7]:
# Read it back in

human_predictions = []
with open('human_out.csv', 'r', encoding="utf-8") as csvfile:
    reader = csv.reader(csvfile)
    for row in reader:
        human_predictions.append(float(row[1]))

In [8]:
def human_pricer(item):
    idx = test.index(item)
    return human_predictions[idx]

In [9]:
human = human_pricer(test[0])
actual = test[0].price
print(f"Human predicted {human} for an item that actually costs {actual}")


Human predicted 120.0 for an item that actually costs 219.0


In [10]:
evaluate(human_pricer, test, size=100)

  0%|          | 0/100 [00:00<?, ?it/s]

$99 $184 $12 $15 $18 $10 $119 $135 $6 $270 $643 $329 $15 $26 $24 $18 $29 $25 $25 $53 $35 $126 $25 $127 $273 $398 $55 $6 $101 $51 $30 $5 $35 $9 $10 $419 $25 $11 $186 $33 $161 $51 $23 $155 $150 $4 $31 $18 $115 $82 $25 $111 $410 $75 $67 $34 $8 $10 $122 $28 $116 $17 $19 $60 $599 $60 $160 $355 $75 $34 $17 $2 $70 $76 $41 $9 $226 $5 $5 $4 $0 $7 $5 $74 $7 $10 $68 $74 $5 $3 $17 $45 $5 $16 $0 $153 $2 $122 $150 $355 

# VANILLA NEURAL NETWORK

In [31]:
# Prepare our documents and prices

y = np.array([float(item.price) for item in train])
documents = [item.summary for item in train]

In [32]:
# Use the HashingVectorizer for a Bag of Words model
# Using binary=True with the CountVectorizer makes "one-hot vectors"

np.random.seed(42)
vectorizer = HashingVectorizer(n_features=5000, stop_words='english', binary=True)
X = vectorizer.fit_transform(documents)

In [33]:
# Define the neural network - here is Pytorch code to create a 8 layer neural network

class NeuralNetwork(nn.Module):
    def __init__(self, input_size):
        super(NeuralNetwork, self).__init__()
        self.layer1 = nn.Linear(input_size, 128)
        self.layer2 = nn.Linear(128, 64)
        self.layer3 = nn.Linear(64, 64)
        self.layer4 = nn.Linear(64, 64)
        self.layer5 = nn.Linear(64, 64)
        self.layer6 = nn.Linear(64, 64)
        self.layer7 = nn.Linear(64, 64)
        self.layer8 = nn.Linear(64, 1)
        self.relu = nn.ReLU()

    def forward(self, x):
        output1 = self.relu(self.layer1(x))
        output2 = self.relu(self.layer2(output1))
        output3 = self.relu(self.layer3(output2))
        output4 = self.relu(self.layer4(output3))
        output5 = self.relu(self.layer5(output4))
        output6 = self.relu(self.layer6(output5))
        output7 = self.relu(self.layer7(output6))
        output8 = self.layer8(output7)
        return output8

In [34]:
# Convert data to PyTorch tensors
X_train_tensor = torch.FloatTensor(X.toarray())
y_train_tensor = torch.FloatTensor(y).unsqueeze(1)

# Split the data into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(X_train_tensor, y_train_tensor, test_size=0.01, random_state=42)

# Create the loader
train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

# Initialize the model
input_size = X_train_tensor.shape[1]
model = NeuralNetwork(input_size)

In [35]:
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Number of trainable parameters: {trainable_params:,}")

Number of trainable parameters: 669,249


In [36]:
# Define loss function and optimizer

loss_function = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# We will do 2 complete runs through the data

EPOCHS = 2

for epoch in range(EPOCHS):
    model.train()
    for batch_X, batch_y in tqdm(train_loader):
        optimizer.zero_grad()

        # The next 4 lines are the 4 stages of training: forward pass, loss calculation, backward pass, optimize
        
        outputs = model(batch_X)
        loss = loss_function(outputs, batch_y)
        loss.backward()
        optimizer.step()

    model.eval()
    with torch.no_grad():
        val_outputs = model(X_val)
        val_loss = loss_function(val_outputs, y_val)

    print(f'Epoch [{epoch+1}/{EPOCHS}], Train Loss: {loss.item():.3f}, Val Loss: {val_loss.item():.3f}')

  0%|          | 0/12375 [00:00<?, ?it/s]

Epoch [1/2], Train Loss: 10079.236, Val Loss: 11701.009


  0%|          | 0/12375 [00:00<?, ?it/s]

Epoch [2/2], Train Loss: 7214.386, Val Loss: 10839.249


In [37]:
def neural_network(item):
    model.eval()
    with torch.no_grad():
        vector = vectorizer.transform([item.summary])
        vector = torch.FloatTensor(vector.toarray())
        result = model(vector)[0].item()
    return max(0, result)

In [38]:
evaluate(neural_network, test)

  0%|          | 0/200 [00:00<?, ?it/s]

$135 $123 $24 $114 $28 $98 $76 $12 $6 $150 $122 $223 $45 $92 $46 $11 $24 $4 $47 $60 $31 $78 $43 $71 $181 $143 $157 $29 $141 $48 $103 $126 $24 $20 $83 $377 $20 $4 $103 $52 $138 $0 $20 $41 $45 $27 $16 $28 $63 $83 $8 $70 $24 $38 $38 $114 $19 $234 $55 $26 $98 $13 $45 $7 $272 $200 $21 $360 $85 $96 $9 $13 $144 $89 $3 $21 $153 $14 $20 $26 $71 $41 $23 $41 $8 $123 $127 $10 $45 $365 $12 $55 $6 $1 $31 $54 $18 $26 $189 $262 $31 $33 $10 $77 $7 $18 $62 $275 $6 $48 $33 $52 $158 $16 $44 $173 $29 $60 $10 $216 $13 $159 $96 $6 $91 $34 $11 $121 $141 $28 $25 $25 $55 $26 $74 $18 $91 $45 $115 $16 $16 $108 $47 $101 $98 $52 $43 $217 $44 $7 $13 $6 $2 $31 $23 $86 $45 $4 $62 $17 $18 $3 $1 $22 $284 $7 $95 $31 $7 $50 $63 $10 $272 $44 $34 $80 $38 $20 $31 $200 $208 $9 $92 $21 $13 $23 $89 $35 $42 $28 $36 $1 $2 $65 $34 $53 $152 $15 $5 $11 

In [66]:
def messages_for(item):
    message = f"Estimate the price of this product. Respond with the price, no explanation\n\n{item.summary}"
    return [
        {"role": "system", "content": "You are a pricing expert. Respond only with a dollar amount like $42.99. No explanation."},
        {"role": "user", "content": message}
    ]

In [67]:
print(test[0].summary)

Title: Excess V2 Distortion/Modulation Pedal  
Category: Music Pedals  
Brand: Old Blood Noise  
Description: A versatile pedal offering distortion and three modulation modes—delay, chorus, and harmonized fifths—with full control over signal routing and expression.  
Details: Features include separate gain, tone, and volume controls; time, depth, and volume per modulation; order switching, soft‑touch bypass, and expression jack for dynamic control.


In [68]:
messages_for(test[0])

[{'role': 'system',
  'content': 'You are a pricing expert. Respond only with a dollar amount like $42.99. No explanation.'},
 {'role': 'user',
  'content': 'Estimate the price of this product. Respond with the price, no explanation\n\nTitle: Excess V2 Distortion/Modulation Pedal  \nCategory: Music Pedals  \nBrand: Old Blood Noise  \nDescription: A versatile pedal offering distortion and three modulation modes—delay, chorus, and harmonized fifths—with full control over signal routing and expression.  \nDetails: Features include separate gain, tone, and volume controls; time, depth, and volume per modulation; order switching, soft‑touch bypass, and expression jack for dynamic control.'}]

# OLLAMA

In [75]:
def ollama_predictor(item):
    response = completion(
        model="ollama/llama3.2:3b",
        messages=messages_for(item),
        api_base="http://localhost:11434"
    )
    return response.choices[0].message.content

In [76]:
ollama_predictor(test[0])

'$199.00'

In [77]:
test[0].price

219.0

In [79]:
evaluate(ollama_predictor, test)

  0%|          | 0/200 [00:00<?, ?it/s]

$11 $66 $29 $60 $20 $187 $109 $95 $6 $1170 $583 $29 $130 $4 $19 $3 $1 $23 $80 $81 $64 $54 $15 $5 $232 $303 $5 $30 $31 $30 $50 $10 $30 $54 $135 $289 $10 $26 $84 $23 $170 $50 $15 $195 $150 $0 $7 $18 $45 $142 $16 $105 $325 $10 $447 $88 $13 $60 $172 $58 $14 $22 $41 $20 $504 $80 $180 $295 $75 $74 $12 $13 $220 $6 $30 $16 $506 $20 $13 $31 $70 $3 $30 $74 $2 $5 $52 $56 $20 $16 $3 $15 $0 $15 $4 $98 $26 $107 $140 $345 $45 $92 $12 $11 $199 $312 $5 $357 $11 $196 $10 $186 $159 $2 $34 $1129 $45 $0 $14 $27 $24 $511 $20 $142 $90 $17 $6 $49 $51 $109 $19 $57 $50 $5 $15 $11 $75 $80 $103 $112 $19 $245 $30 $10 $124 $7 $9 $85 $155 $3 $1 $6 $17 $10 $6 $209 $1 $41 $980 $80 $40 $17 $17 $12 $160 $17 $2 $20 $10 $15 $5 $3 $90 $7 $37 $1 $42 $43 $6 $63 $54 $35 $130 $19 $80 $18 $33 $73 $30 $1 $15 $49 $95 $161 $30 $50 $30 $150 $26 $4 

# FORNTIER

In [80]:
# def claude_opus_4_5(item):
#     response = completion(model="anthropic/claude-opus-4-5", messages=messages_for(item))
#     return response.choices[0].message.content

In [81]:
# evaluate(claude_opus_4_5, test)